# **Import des modules nécessaires**

In [ ]:
import pandas as pd #pour la manipulation de données
import numpy as np
from pathlib import Path #pour la gestion des chemins de fichiers
import spacy #pour le prétraitement de texte
from spacy.lang.fr.stop_words import STOP_WORDS as spacy_stopwords

from bertopic import BERTopic #pour la modélisation de sujets
from bertopic.vectorizers import ClassTfidfTransformer #pour la vectorisation de texte spécifique à BERTopic
from sentence_transformers import SentenceTransformer #pour les embeddings de phrase

from umap import UMAP #pour la réduction de dimensionnalité
from hdbscan import HDBSCAN #pour le clustering de BERTopic

from sklearn.feature_extraction.text import CountVectorizer #pour la vectorisation de texte
from sklearn.metrics import silhouette_score

from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel


# **Chargement du corpus de Zola et de spacy**


In [2]:
df=pd.read_csv(Path("..") /"data" /"2_processed"/ "02_corpus_zola.csv", encoding="utf-8",)
df.head()


,roman,annee,ordre_romans,paquet_id,texte,nb_mots
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",214
1,1865 La confession de Claude.,1865,1,2,"La mansarde entière me réclame les rires, les ...",187
2,1865 La confession de Claude.,1865,1,3,Le grillon chantait; le souffle harmonieux des...,146
3,1865 La confession de Claude.,1865,1,4,"brunes et rieuses filles, étaient reines des m...",190
4,1865 La confession de Claude.,1865,1,5,"Pars cependant, puisque tu as soif de la vie. ...",126


In [3]:
df.shape

(21260, 6)

# **Traitement du Corpus de Zola**

In [ ]:
stop_perso = {
    "grand", "petit", "homme", "femme", "jour", "heure", "coup", "œil", "oeil", 
    "main", "bras", "tête", "voix", "milieu", "eau", "terre", "air", "monde", 
    "chose", "nuit", "vie", "enfant", "père", "mère", "fille", "garçon", 
    "monsieur", "madame", "falloir", "aller", "voir", "dire", "faire", 
    "pouvoir", "vouloir", "savoir", "venir", "devoir", "prendre", "donner",
    "oui", "non", "où", "quand", "comment", "bon", "jeune", "vieux", "suite"
}

# Chargement du modèle avec désactivation du 'parser' syntaxique pour gagner en vitesse
# On garde impérativement 'ner' pour repérer les personnages/lieux et 'lemmatizer'
nlp = spacy.load("fr_core_news_lg", disable=["parser"])
nlp.max_length = 2_000_000  

#on convertit en liste
textes_bruts = df["texte"].astype(str).tolist()

textes_nettoyes = []

# Utilisation de nlp.pipe pour traiter les textes par blocs (très rapide)
for doc in nlp.pipe(textes_bruts, batch_size=256, n_process=2): 
    tokens = [] # Liste pour stocker les tokens nettoyés
    
    for token in doc:
        lemme = token.lemma_.lower() # Obtenir le lemme du token en minuscules
        
        if (
            not token.is_stop # Ignorer les stop words spaCy par défaut
            and not token.is_punct # Ignorer la ponctuation
            and not token.like_num # Ignorer les chiffres
            and not token.is_space # Ignorer les espaces vides
            and token.ent_type_ not in ['PER', 'LOC', 'ORG'] # Ignorer les Personnages, Lieux et Organisations
            and token.pos_ in {"NOUN", "ADJ"}  # Garder Noms, Adjectifs ET Verbes
            and len(lemme) > 2 # Ignorer les mots de 1 ou 2 lettres
            and lemme not in stop_perso
        ):
            tokens.append(lemme)
            
    # Rejoindre les tokens validés et les ajouter à la liste finale
    textes_nettoyes.append(" ".join(tokens))

# Application de la liste nettoyée à la nouvelle colonne du DataFrame
df["phrases_lemm"] = textes_nettoyes

# Affichage du résultat
df[["phrases_lemm"]].head()

,phrases_lemm
0,hiver matin frais manteau brouillard saison so...
1,mansarde entier rire richesse sœur foyer feu j...
2,grillon souffle harmonieux causerie lèvre cœur...
3,brun rieur moisson vendange épi grappe sentier...
4,soif projet soi ferme loyal action rêve vis gr...


## **1) Choix du modèle d'embedding**

Ici je vais choisir un modèle d'embedding pré-entraîné pour transformer les textes en vecteurs numériques. Je vais utiliser un modèle de la bibliothèque Sentence Transformers, qui est compatible avec BERTopic.

In [55]:
embedding_model = SentenceTransformer(
    "dangvantuan/sentence-camembert-base"
)

print("Génération des embeddings sémantiques...")
embeddings = embedding_model.encode(df['texte'].tolist(),batch_size=64, show_progress_bar=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Génération des embeddings sémantiques...


Batches:   0%|          | 0/333 [00:00<?, ?it/s]

## **2) Pipeline de Traitement**

### 1) HDBSCAN et UMAP

### 3) CountVectorizer et ClassTfidfTransformer avec des stop words personnalisés 

In [58]:
hdbscan_model = HDBSCAN( min_cluster_size=27, 
                        min_samples=4, 
                        metric='euclidean', 
                        cluster_selection_method='eom',
                        prediction_data=True)

umap_model = UMAP( n_neighbors=20,
                  n_components=3, 
                  min_dist=0.0, 
                  metric="cosine",
                  random_state=42)


vectorizer_model = CountVectorizer(
    min_df=2,    # Le mot doit apparaître dans au moins 2 segments pour être pris en compte (élimine les fautes ou mots uniques)
    max_df=0.6)

ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True
)


topic_model = BERTopic(
    language="french",
    hdbscan_model=hdbscan_model,
    umap_model=umap_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=False,
    verbose=True,
    #nr_topics="auto"
)
topics, probs = topic_model.fit_transform(df["phrases_lemm"].tolist(), embeddings= embeddings)

new_topics = topic_model.reduce_outliers(
    df["phrases_lemm"].tolist(), 
    topics, 
    strategy="embeddings",
    embeddings=embeddings
)

# Met à jour le modèle avec ces nouveaux thèmes plus propres
topic_model.update_topics(df["phrases_lemm"].tolist(), topics=new_topics)

2026-07-09 14:53:00,249 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-09 14:53:10,503 - BERTopic - Dimensionality - Completed ✓
2026-07-09 14:53:10,504 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-09 14:53:10,771 - BERTopic - Cluster - Completed ✓
2026-07-09 14:53:10,773 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-09 14:53:11,008 - BERTopic - Representation - Completed ✓
2026-07-09 14:53:11,214 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


## **3) Topics Présent**

In [59]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,0,1552,0_soleil_haut_ciel_blanc,"[soleil, haut, ciel, blanc, noir, ombre, long,...",[bout forêt endroit mur feuille fouillis impén...
1,1,1061,1_amour_cœur_chair_amant,"[amour, cœur, chair, amant, pensée, tendresse,...",[fond stupeur souvenir éclair brusque mémoire ...
2,2,635,2_lit_chambre_douleur_porte,"[lit, chambre, douleur, porte, sang, minute, m...",[peur corps mou lâche déchirement profond appr...
3,3,496,3_cœur_heureux_bonheur_larme,"[cœur, heureux, bonheur, larme, amour, écoute,...",[arbre poison bête taillis noir venin vipère r...
4,4,370,4_armée_soldat_général_prussien,"[armée, soldat, général, prussien, troupe, ins...",[camp lutte suprême front attaque kilomètre ce...
...,...,...,...,...,...
82,82,207,82_frère_modèle_pape_cardinal,"[frère, modèle, pape, cardinal, doute, justice...",[immobile extase œuvre corniche fresque prêtre...
83,83,176,83_rire_nez_rue_verre,"[rire, nez, rue, verre, fois, matin, oreille, ...",[ridicule bal point orchestre premier apitoiem...
84,84,96,84_cuisine_mot_porte_maison,"[cuisine, mot, porte, maison, lit, mur, fenêtr...",[argent soir queue pareil remède pis tour carr...
85,85,107,85_porte_comte_tour_face,"[porte, comte, tour, face, dépêche, soir, affa...",[tendre prévenance intention meilleur morceau ...


In [ ]:
# 1. Préparer les données pour Gensim (une liste de listes de mots)
# On suppose que df["phrases_lemm"] contient tes textes nettoyés
textes_tokenises = [texte.split() for texte in df["phrases_lemm"].tolist()]
dictionnaire = Dictionary(textes_tokenises)
corpus = [dictionnaire.doc2bow(texte) for texte in textes_tokenises]

# 2. Extraire les mots-clés des topics trouvés par BERTopic
# On ignore le topic -1 s'il existe
topics_mots = []
for topic_id in set(topics):
    if topic_id != -1:
        # Récupère juste les mots, pas les scores
        mots = [mot[0] for mot in topic_model.get_topic(topic_id)]
        topics_mots.append(mots)

# 3. Calculer le score de cohérence (C_v)
cm = CoherenceModel(
    topics=topics_mots, 
    texts=textes_tokenises, 
    corpus=corpus, 
    dictionary=dictionnaire, 
    coherence='c_v'
)
coherence_score = cm.get_coherence()
print(f"Score de Cohérence (C_v) : {coherence_score:.4f}")

Score de Cohérence (C_v) : 0.5454


In [ ]:
# On récupère les documents qui NE SONT PAS dans le topic -1 (outliers)
# (Si tu as utilisé reduce_outliers, tous les documents auront un topic valid)
indices_valides = [i for i, topic in enumerate(topics) if topic != -1]

# On filtre les embeddings et les topics
embeddings_valides = np.array([embeddings[i] for i in indices_valides])
topics_valides = [topics[i] for i in indices_valides]

# Calcul du score
score = silhouette_score(embeddings_valides, topics_valides)
print(f"Score de Silhouette : {score:.3f}")

Score de Silhouette : -0.004


In [54]:
fig = topic_model.visualize_hierarchy()
fig.show()

In [46]:
# Récupération de la dimension temporelle
timestamps = df['annee'].tolist()

# Génération des topics dans le temps
topics_over_time = topic_model.topics_over_time(
    df['phrases_lemm'].tolist(), 
    timestamps, 
    nr_bins=15
)

topic_model.visualize_topics_over_time(topics_over_time) #topics=themes_interet)

15it [00:02,  5.34it/s]


In [33]:
for topic_id in topic_info["Topic"].head(15):
    if topic_id != -1:
        print("\nTOPIC", topic_id)
        print(topic_model.get_topic(topic_id)[:15])


TOPIC 0
[('amour', np.float64(0.009174945649589763)), ('cœur', np.float64(0.007893323491976264)), ('pensée', np.float64(0.006963188531995426)), ('tendresse', np.float64(0.006776902520647584)), ('chair', np.float64(0.006627302695977597)), ('amant', np.float64(0.006502291803195249)), ('passion', np.float64(0.006155300405704173)), ('désir', np.float64(0.005976537925860696)), ('joie', np.float64(0.0056692285968002396)), ('mort', np.float64(0.0055830577910317295))]

TOPIC 1
[('larme', np.float64(0.010071346090276484)), ('écoute', np.float64(0.009841280782517172)), ('cœur', np.float64(0.009545527385428267)), ('heureux', np.float64(0.009187803974782494)), ('chéri', np.float64(0.009040064322841697)), ('bonheur', np.float64(0.008838926786177422)), ('raison', np.float64(0.008754754659666615)), ('pauvre', np.float64(0.008088457783283102)), ('amour', np.float64(0.007804767212940487)), ('tai', np.float64(0.0071474555683395695))]

TOPIC 2
[('fortune', np.float64(0.007438861500858023)), ('fils', np.